In [1]:
%reset -f
%reload_ext autoreload
%autoreload 2

In [2]:
import os,sys
current_path = os.getcwd()
sys.path.append("/home/zhuchen/poc/scorecard") 
sys.path.append("/home/zhuchen/poc/02-xyf/config") 

In [3]:
name = current_path.split('/')[-1].split('_')
data = name[0]
cust = name[1]

In [4]:
print(name)

['deltaV1', 'all']


In [5]:
FILE_PATH = f'/home/zhuchen/poc/02-xyf/{data}_{cust}/file/'
DATA_PATH = f'/home/zhuchen/poc/02-xyf/{data}_{cust}/data/'
TMP_PATH  = f'/home/zhuchen/poc/02-xyf/{data}_{cust}/tmp/'
DATA_OR_PATH = '/home/zhuchen/poc/02-xyf/data_or/'

In [6]:
import gc
import pandas as pd 
import numpy as np  
import math  
import matplotlib.pyplot as plt
import copy
import seaborn as sns
from pylab import mpl
import temp_config


isExists=os.path.exists(FILE_PATH)
if not isExists:
    os.makedirs(FILE_PATH) 
    
isExists=os.path.exists(DATA_PATH)
if not isExists:
    os.makedirs(DATA_PATH) 

import warnings
warnings.filterwarnings("ignore")

In [7]:
# config cell
start_seed = temp_config.start_seed
end_seed = temp_config.end_seed
print("start_seed, end_seed \n",start_seed, end_seed)

start_seed, end_seed 
 100 110


In [8]:
# data_all = pd.read_pickle(DATA_PATH + 'data_model.pkl')

### 树模型

In [9]:
import gc
gc.collect()

0

In [10]:
# test_dtata = pd.read_csv(DATA_PATH + 'data_ft_2.csv')

In [11]:
# test_dtata.head()

In [12]:
from DataPreprocessing.Datasets.Split import SampleChoose

# 防止内存
import gc
gc.collect()

ex_lst = ['mobile','backPointTime','qudao3_act','label',
 'weight',
 'target',
 'month_time',
 'day_time',
 'week_time',]
# ft_lst = [i for i in data_all.columns if i not in ex_lst]
# len(ft_lst)

cs = SampleChoose(DATA_PATH + 'data_ft_2.csv',
                  DATA_PATH + 'sample_choose_500_1.csv',
                  'label', ex_lst)

cs.find(start_seed, end_seed, 2, model_switch=[0, 1, 0])
cs.save(None)

当前内存占用: 643.39 MB


100%|██████████| 66/66 [00:00<00:00, 270.88it/s]


最终内存占用: 343.63 MB
下降了 46.6%
100 =========================
{'train_auc_xgb': 0.5679739503355655, 'train_ks_xgb': 0.09488777767348744, 'valid_auc_xgb': 0.5661081321852938, 'valid_ks_xgb': 0.09189594294872205, 'oot_auc_xgb': 0.5337293719230036, 'oot_ks_xgb': 0.05496053884100238}
102 =========================
{'train_auc_xgb': 0.5687158952096709, 'train_ks_xgb': 0.09550386104266873, 'valid_auc_xgb': 0.5655159807607923, 'valid_ks_xgb': 0.0891209398520616, 'oot_auc_xgb': 0.5344543205788745, 'oot_ks_xgb': 0.054705923199158124}
104 =========================
{'train_auc_xgb': 0.5694118846321204, 'train_ks_xgb': 0.09685778468078743, 'valid_auc_xgb': 0.5624470776641377, 'valid_ks_xgb': 0.08832661076933568, 'oot_auc_xgb': 0.5351086142561798, 'oot_ks_xgb': 0.0585826303588024}
106 =========================
{'train_auc_xgb': 0.5694213860208905, 'train_ks_xgb': 0.0950808472149911, 'valid_auc_xgb': 0.5638081358222775, 'valid_ks_xgb': 0.09027993631828468, 'oot_auc_xgb': 0.5353880329662797, 'oot_ks_xgb':

In [13]:
# res = pd.read_csv(DATA_PATH + 'cs_seed.csv')

In [14]:
cs.res

,train_auc_xgb,train_ks_xgb,valid_auc_xgb,valid_ks_xgb,oot_auc_xgb,oot_ks_xgb,flag
0,0.567974,0.094888,0.566108,0.091896,0.533729,0.054961,100
1,0.568716,0.095504,0.565516,0.089121,0.534454,0.054706,102
2,0.569412,0.096858,0.562447,0.088327,0.535109,0.058583,104
3,0.569421,0.095081,0.563808,0.090280,0.535388,0.058230,106
4,0.568200,0.094361,0.566564,0.094243,0.533460,0.055979,108


In [15]:
def get_res(cs, model_name='xgb'):
    if f'oot_ks_{model_name}' in cs.res.columns:
        res = cs.res[cs.res[f'valid_ks_{model_name}'] > 0.0][['flag', f'train_ks_{model_name}', f'valid_ks_{model_name}', f'oot_ks_{model_name}']]
        res = res[res[f'train_ks_{model_name}'] > 0.0]
        res = res[res[f'oot_ks_{model_name}'] > 0.0]
        res['delta'] = res[f'train_ks_{model_name}'] - res[f'oot_ks_{model_name}']
        res['delta1'] = res[f'valid_ks_{model_name}'] - res[f'oot_ks_{model_name}']
        res['delta2'] = res[f'valid_ks_{model_name}'] - res[f'train_ks_{model_name}']
        res = res[abs(res['delta']) < 1]
        res = res[abs(res['delta1']) < 1]
        res = res[abs(res['delta2']) < 1]
        res.sort_values(by=f'oot_ks_{model_name}', ascending=False, inplace=True)
        # res.sort_values(by=f'valid_ks_{model_name}', ascending=False, inplace=True)
    else:
        # 这里为什么会有没有oot的情况？
        res = cs.res[cs.res[f'valid_ks_{model_name}'] > 0.22][['flag', f'train_ks_{model_name}', f'valid_ks_{model_name}']]
        res = res[res[f'train_ks_{model_name}'] > 0.22]
        res['delta2'] = res[f'valid_ks_{model_name}'] - res[f'train_ks_{model_name}']
        res = res[abs(res['delta2']) < 0.03]
        res.sort_values(by=f'valid_ks_{model_name}', ascending=False, inplace=True)
    return res

In [16]:
res_xgb = get_res(cs,model_name = 'xgb')
# res_lgb = get_res(cs,model_name = 'lgb')

In [17]:
res_xgb # 434

,flag,train_ks_xgb,valid_ks_xgb,oot_ks_xgb,delta,delta1,delta2
2,104,0.096858,0.088327,0.058583,0.038275,0.029744,-0.008531
3,106,0.095081,0.090280,0.058230,0.036851,0.032050,-0.004801
4,108,0.094361,0.094243,0.055979,0.038382,0.038263,-0.000119
0,100,0.094888,0.091896,0.054961,0.039927,0.036935,-0.002992
1,102,0.095504,0.089121,0.054706,0.040798,0.034415,-0.006383


In [18]:
# res_lgb #2872

In [19]:
# res.sort_values(by='valid_ks_xgb', ascending=False, inplace=True)

In [20]:
res_xgb.to_csv(DATA_PATH + 'cs_seed_xgb.csv')
# res_lgb.to_csv(DATA_PATH + 'cs_seed_lgb.csv')

In [21]:
best_flag = res_xgb.loc[res_xgb['oot_ks_xgb'].idxmax(), 'flag']
print(f"Best flag: {best_flag}")
# 保存best_flag，处理标量值的情况
if best_flag is not None:
    pd.DataFrame({'best_flag': [best_flag]}).to_csv(FILE_PATH + 'best_flag.csv', index=False)
else:
    pd.DataFrame({'best_flag': ['None']}).to_csv(FILE_PATH + 'best_flag.csv', index=False)


Best flag: 104


In [22]:
%reset -f